# 01 — Explore the RUKOPYS Dataset

**Purpose:** Understand what the dataset contains before writing any training code.

**What you will do here:**
1. Load a few samples from the dataset (streaming — no full download needed).
2. Inspect every field in a sample record.
3. Understand what `regions`, `bbox`, `type`, `language`, `legibility`, and `text` mean.
4. Confirm the dataset loads correctly on your machine.

> Run cells top-to-bottom. Every cell has a comment explaining what it does.

In [ ]:
# ── Setup: make sure the project root is on sys.path ─────────────
import sys
from pathlib import Path

# Go up one level from notebooks/ to reach project root
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ── Option A: Stream (no download, limited RAM machines) ──────────
# Streaming reads one record at a time from Hugging Face.
# Use this if you have limited disk space or a slow connection.

from src.data.load_dataset import stream_split

samples = []
for i, sample in enumerate(stream_split('train')):
    samples.append(sample)
    if i >= 4:   # grab 5 samples
        break

print(f'Loaded {len(samples)} samples via streaming')
print(f'Keys in a sample: {list(samples[0].keys())}')

In [ ]:
# ── Option B: Full load (faster indexing, needs disk space) ───────
# Uncomment this block and comment out Option A if you want to
# cache the full dataset locally.

# from src.data.load_dataset import load_split
# ds = load_split('train')
# print(f'Loaded {len(ds):,} samples')
# samples = [ds[i] for i in range(5)]

In [ ]:
# ── Inspect Sample 0 in detail ────────────────────────────────────
# This calls our inspect_dataset module which pretty-prints every field.

from src.data.inspect_dataset import inspect_sample

inspect_sample(samples[0], sample_idx=0)

In [ ]:
# ── Manually explore the regions list ────────────────────────────
# A 'region' is one annotated area on the document page.
# Let's look at the raw region data to understand the structure.

import json

sample = samples[0]
regions = sample.get('regions', [])
if isinstance(regions, str):
    regions = json.loads(regions)

print(f'This document has {len(regions)} regions.\n')

for i, r in enumerate(regions):
    print(f'Region [{i}]')
    print(f'  bbox       : {r.get("bbox")}')
    print(f'  type       : {r.get("type")}')
    print(f'  language   : {r.get("language")}')
    print(f'  legibility : {r.get("legibility")}')
    text = r.get('text', '')
    print(f'  text       : "{text[:80]}{"..." if len(text) > 80 else ""}"')
    print()

In [ ]:
# ── What is a "scorable" region? ──────────────────────────────────
# The competition only scores regions where:
#   - language = 'uk'       (Ukrainian)
#   - legibility = 'legible'
#   - type NOT IN image, graph
#   - text is non-empty

scorable = [
    r for r in regions
    if r.get('language') == 'uk'
    and r.get('legibility') == 'legible'
    and r.get('type') not in ('image', 'graph')
    and r.get('text', '').strip()
]

print(f'Total regions    : {len(regions)}')
print(f'Scorable regions : {len(scorable)}')
print()
for r in scorable[:3]:
    print(f'  type={r["type"]}  text="{r["text"][:60]}"')

In [ ]:
# ── Show the image (if available) ────────────────────────────────
# When loaded via HF datasets (not streaming from JSONL),
# the 'image' key contains a PIL Image object you can display here.

from IPython.display import display

img = sample.get('image')
if img is not None:
    print(f'Image mode: {img.mode}, size: {img.size}')
    display(img.resize((400, int(400 * img.size[1] / img.size[0]))))
else:
    print('Image not available in this mode (streaming from JSONL).')
    print(f'image_width={sample.get("image_width")}  image_height={sample.get("image_height")}')

In [ ]:
# ── Quick look at 5 different documents ──────────────────────────
print('=== 5 Sample Documents ===')
for i, s in enumerate(samples):
    regs = s.get('regions', [])
    if isinstance(regs, str):
        regs = json.loads(regs)
    types = [r.get('type') for r in regs]
    print(f'[{i}] {s.get("file_name"):40s}  '
          f'source={s.get("source"):12s}  '
          f'regions={len(regs)}  '
          f'types={set(types)}')

## ✅ Next Step

```bash
# Save metadata locally so future notebooks work offline:
python -m src.data.load_dataset --splits train test --save-jsonl

# Then open:
# notebooks/02_visualize_annotations.ipynb
```